<a href="https://colab.research.google.com/github/MoEissa140/FlyRank-Intern/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoEissa140/FlyRank-Intern/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
!pip install -q huggingface_hub duckdb
import duckdb, os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
import pandas as pd

In [ ]:
# Reuse the connection/secret setup from w03 if this is a fresh runtime:
# con = duckdb.connect(); con.execute("INSTALL httpfs; LOAD httpfs;")
# con.execute(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{os.environ[\"HF_TOKEN\"]}');")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

base = "hf://datasets/FlyRank/internship-warehouse"

feature_vector = con.sql(f"""
WITH feat_window AS (
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_impressions) AS avg_impressions,
        AVG(gsc_clicks) AS avg_clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_engaged_sessions) * 1.0 / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,
        SUM(scroll_events) * 1.0 / NULLIF(SUM(ga4_pageviews), 0) AS scroll_rate,
        BOOL_OR(ga4_data_available) AS has_ga4
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id, client_hash_id
),
target_window AS (
    SELECT
        content_hash_id,
        AVG(gsc_impressions) AS avg_impressions_next
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-04/*.parquet')
    GROUP BY content_hash_id
)
SELECT
    f.*,
    t.avg_impressions_next,
    CASE WHEN t.avg_impressions_next < f.avg_impressions THEN 1 ELSE 0 END AS is_declining_label
FROM feat_window f
JOIN target_window t USING (content_hash_id)
WHERE f.avg_impressions > 0
""").df()

# fill / clean
feature_vector["engagement_rate"] = feature_vector["engagement_rate"].fillna(0)
feature_vector["scroll_rate"] = feature_vector["scroll_rate"].fillna(0)
feature_vector["avg_position"] = feature_vector["avg_position"].fillna(feature_vector["avg_position"].median())

# categorical handling: bucket avg_position into a tier
feature_vector["position_tier"] = pd.cut(
    feature_vector["avg_position"], bins=[0,3,10,20,100], labels=["top3","page1","page2","beyond"]
)

feature_vector.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,avg_impressions,avg_clicks,avg_position,engagement_rate,scroll_rate,has_ga4,avg_impressions_next,is_declining_label,position_tier
0,content_76c1f31e2b38f054,client_62f4a7e64f5e0096,24.096774,0.032258,29.377302,0.0,0.0,<NA>,8.300000,1,beyond
1,content_ffc5ab4b34aab1f8,client_62f4a7e64f5e0096,16.161290,0.064516,16.954341,0.0,0.0,<NA>,9.366667,1,page2
2,content_9739856fc83dc1ca,client_62f4a7e64f5e0096,93.322581,0.032258,9.445005,0.0,0.0,<NA>,24.066667,1,page1
3,content_50266f97d6233542,client_62f4a7e64f5e0096,0.258065,0.000000,24.000000,0.0,0.0,<NA>,0.500000,0,beyond
4,content_3d1dc691a3502105,client_62f4a7e64f5e0096,224.354839,0.290323,4.365083,0.0,0.0,<NA>,102.266667,1,page1


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available before decision point? |
|---|---|---|---|
| `avg_impressions` | mean daily GSC impressions, prior 90-day window | rows filtered to >0, no fill needed | Yes — trailing window only |
| `avg_clicks` | mean daily GSC clicks, prior 90-day window | 0 is valid (no clicks), no fill needed | Yes |
| `avg_position` | mean daily average search position | filled with column median where GSC unavailable | Yes |
| `engagement_rate` | engaged sessions / total sessions, prior window | filled with 0 where GA4 unavailable (documented limit, not "no engagement") | Yes |
| `scroll_rate` | scroll events / pageviews, prior window | filled with 0 where GA4 unavailable | Yes |
| `position_tier` | categorical bucket of `avg_position` | derived, no separate missingness | Yes |

In [ ]:
feature_vector[["avg_impressions","avg_clicks","avg_position","engagement_rate","scroll_rate"]].isna().sum()

,0
avg_impressions,0
avg_clicks,0
avg_position,0
engagement_rate,0
scroll_rate,0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

X_cols = ["avg_impressions","avg_clicks","avg_position","engagement_rate","scroll_rate"]
y = feature_vector["is_declining_label"]
groups = feature_vector["client_hash_id"]

def honest_auc(df, cols):
    X = df[cols].fillna(0)
    gkf = GroupKFold(n_splits=5)
    aucs = []
    for train_idx, test_idx in gkf.split(X, y, groups):
        m = LogisticRegression(max_iter=1000).fit(X.iloc[train_idx], y.iloc[train_idx])
        p = m.predict_proba(X.iloc[test_idx])[:,1]
        aucs.append(roc_auc_score(y.iloc[test_idx], p))
    return sum(aucs)/len(aucs)

print("Honest AUC (no leak):", honest_auc(feature_vector, X_cols))

# Now attack it — inject a feature derived straight from the label/target window
feature_vector["leaky_future_impressions"] = feature_vector["avg_impressions_next"]
print("Leaked AUC:", honest_auc(feature_vector, X_cols + ["leaky_future_impressions"]))

# remove it, confirm honest number is restored
feature_vector = feature_vector.drop(columns=["leaky_future_impressions"])
print("Confirmed honest AUC again:", honest_auc(feature_vector, X_cols))

Honest AUC (no leak): 0.5388627257283698
Leaked AUC: 0.9999926831856302
Confirmed honest AUC again: 0.5388627257283698


## 4. What I excluded and why

| Excluded field | Why |
|---|---|
| `health_score`, `priority_score`, `action_type` | Not present in this dataset at all (confirmed via `DESCRIBE` in ML-04) — but flagging explicitly: even if they existed, they're FlyRank product decision outputs, not observable signals, and using them would cause a circular result |
| `avg_impressions_next` (target-window impressions) | Directly derived from the label window — this is the leak I demonstrated and removed in Section 3 |
| `client_hash_id`, `content_hash_id` | Join keys only, carry no predictive signal on their own — used for grouping/validation, not as model inputs |
| `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, etc. | Excluded from this lane's feature set — extremely sparse (documented in ML-04: ~4% GA4 coverage overall, AI referral rows sparser still), belongs to the separate AI Referral freestyle direction, not Refresh Scoring |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.